# N5 — Joint decoding and MIDI export

In [ ]:
from pathlib import Path
import sys, yaml, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
CFG=yaml.safe_load((ROOT/'config.yaml').read_text())
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('device:',DEVICE,'debug:',CFG['data']['debug'])

from piano_transcriber.decoding import threshold_decode,rbm_gibbs_decode,autoregressive_beam_decode
from piano_transcriber.labels import roll_to_notes,notes_to_midi_bytes
from piano_transcriber.priors import BernoulliRBM,NADE,MADE

Use real cached acoustic frame probabilities for research evaluation. The random array below is a decoder smoke test only.

In [ ]:
prob=np.random.default_rng(4).uniform(.01,.99,(100,88)).astype('float32')
base=threshold_decode(prob,.5);print(base.shape,base.sum())
rbm=BernoulliRBM(88,128).to(DEVICE)
print(rbm_gibbs_decode(prob,rbm,.5,4).shape)
nade=NADE(88,128).to(DEVICE);made=MADE(88,(128,128)).to(DEVICE)
print(autoregressive_beam_decode(prob[:2],nade,.5,8).shape)
print(autoregressive_beam_decode(prob[:2],made,.5,8).shape)

In [ ]:
notes=roll_to_notes(base,31.25,21,.5,2);midi=notes_to_midi_bytes(notes)
out=ROOT/'reports'/'decoder_smoke_test.mid';out.write_bytes(midi)
print('events',len(notes),'MIDI bytes',len(midi))

Compare threshold, RBM Gibbs and NADE/MADE beam outputs on identical clips. Report decoding runtime separately from acoustic runtime.